In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime
import time
class OverAllState(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str

# runtime.stream_writer 实现流式输出内容
def node_a(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("节点 A 正在执行...")
    time.sleep(2)
    return {
        "node_a_output": "节点A的输出"
    }

def node_b(state: OverAllState, runtime: Runtime) -> OverAllState:
    stream_writer = runtime.stream_writer
    stream_writer("节点 B 正在执行...")
    time.sleep(2)
    return {
        "node_b_output": "节点B的输出"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
for chunk in graph.stream(
    {"initial_state": "初始状态"},
    # 如果想要用户结点输出流式内容，必须添加 stream_mode = ["custom"]
    stream_mode=["custom"],
):
    print(chunk)

('custom', '节点 A 正在执行...')
('custom', '节点 B 正在执行...')
